# Section 1 - Static TLE Analysis

In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import json
import os
from astropy.time import Time
from datetime import datetime, timezone

print("Imports loaded!")

Imports loaded!


In [2]:
# Cell 2 — Load all caches
TLE_CACHE_FILE    = os.path.expanduser("~/tle_cache.json")
SATCAT_CACHE_FILE = os.path.expanduser("~/satcat_cache.json")
PROP_CACHE_FILE   = os.path.expanduser("~/propagation_cache.npz")

# TLE and SATCAT
with open(TLE_CACHE_FILE) as f:
    tle_cache = json.load(f)
with open(SATCAT_CACHE_FILE) as f:
    satcat_cache = json.load(f)

tles = tle_cache['tles']

# Propagation
prop = np.load(PROP_CACHE_FILE)
r_teme = prop['r_teme']
v_teme = prop['v_teme']
e      = prop['e']
lats   = prop['lats']
lons   = prop['lons']
alts   = prop['alts']
times  = Time(prop['times_jd1'], prop['times_jd2'], format='jd', scale='utc')

print(f"All caches loaded!")
print(f"   Launch:      {satcat_cache['intldes']}")
print(f"   Satellites:  {len(tles)}")
print(f"   Timesteps:   {r_teme.shape[1]}")
print(f"   Time range:  {times[0].iso} → {times[-1].iso} UTC")

All caches loaded!
   Launch:      2023-174
   Satellites:  55
   Timesteps:   2000
   Time range:  2026-04-22 04:31:33.481 → 2026-04-23 04:31:33.481 UTC


In [3]:
def parse_bstar(s):
    s = s.strip()
    # Find the exponent sign
    for i in range(1, len(s)):
        if s[i] in '+-':
            mantissa = float('0.' + s[:i])
            exp = int(s[i:])
            return mantissa * (10 ** exp)
    return float(s)

In [4]:
# Cell 4 — Parse TLE metadata into dataframe
rows = []
for i, t in enumerate(tles):
    line1 = t['line1']
    line2 = t['line2']
    
    intldes_raw = line1[9:17].strip()
    year  = intldes_raw[:2]
    num   = intldes_raw[2:5]
    piece = intldes_raw[5:]
    full_year = f"19{year}" if int(year) >= 57 else f"20{year}"
    intldes = f"{full_year}-{num}{piece}"
    
    rows.append({
        'idx':        i,
        'name':       t['name'],
        'intldes':    intldes,
        'norad':      line1[2:7].strip(),
        'epoch':      line1[18:32].strip(),
        'bstar': parse_bstar(line1[53:61]),
        'mean_motion':float(line2[52:63].strip()),
        'inclination':float(line2[8:16].strip()),
        'eccentricity':float('0.' + line2[26:33].strip()),
        'raan':       float(line2[17:25].strip()),
        'arg_perigee':float(line2[34:42].strip()),
        'mean_anomaly':float(line2[43:51].strip()),
    })

df = pd.DataFrame(rows)

# Add propagated stats
df['alt_mean'] = [alts[i][e[i]==0].mean() for i in range(len(tles))]
df['alt_min']  = [alts[i][e[i]==0].min()  for i in range(len(tles))]
df['alt_max']  = [alts[i][e[i]==0].max()  for i in range(len(tles))]
df['period_min'] = 24*60 / df['mean_motion']

print(f"Dataframe built — {len(df)} satellites")
df

Dataframe built — 55 satellites


,idx,name,intldes,norad,epoch,bstar,mean_motion,inclination,eccentricity,raan,arg_perigee,mean_anomaly,alt_mean,alt_min,alt_max,period_min
0,0,ORBASTRO-PC-1,2023-174C,58258,26111.93675734,0.000441,15.461509,97.3772,0.000628,196.8750,26.4121,333.7447,439.795821,425.981686,458.352155,93.134507
1,1,LEMUR 2 BASS,2023-174D,58259,26111.82374828,0.000351,15.412793,97.3838,0.000692,195.8311,25.6275,334.5313,454.493180,439.899137,472.850292,93.428880
2,2,PLATERO,2023-174G,58262,26111.51551040,0.000320,15.597044,97.3914,0.000344,207.1070,315.7013,44.3968,400.477548,389.510744,415.130969,92.325184
3,3,BRO-11,2023-174J,58264,26111.95961049,0.000163,15.268038,97.3911,0.001157,189.3405,53.3780,306.8521,497.633607,481.060712,520.299969,94.314671
4,4,FLOCK 4Q 1,2023-174Q,58270,26111.81108734,0.000703,15.706942,97.3817,0.000388,203.2122,344.4425,15.6718,368.386912,356.402598,384.384637,91.679207
5,5,FLOCK 4Q 16,2023-174R,58271,26111.67132329,0.000378,15.468483,97.3818,0.000563,198.5804,20.7435,339.4042,437.752313,424.320800,455.666617,93.092515
6,6,TIGER-6,2023-174S,58272,26111.81639113,0.001056,15.814453,97.3866,0.000355,207.7496,266.0566,94.0297,337.286578,327.187743,351.840509,91.055946
7,7,FLOCK 4Q 11,2023-174T,58273,26111.79639681,0.000371,15.472940,97.3869,0.000529,199.2408,18.7272,341.4171,436.495213,423.254836,454.193984,93.065700
8,8,FLOCK 4Q 36,2023-174U,58274,26111.14810975,0.000570,15.539808,97.3845,0.000517,201.6364,353.5284,6.5902,416.880764,403.866223,432.772357,92.665238
9,9,FLOCK 4Q 12,2023-174V,58275,26111.76768088,0.001124,15.600343,97.3828,0.000263,202.3922,310.9589,49.1440,399.224893,388.454527,414.344950,92.305665


In [5]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['inclination'],
    nbinsx=20,
    marker_color='cyan',
    marker_line=dict(color='black', width=1),
    name='Inclination'
))

fig.update_layout(
    title=f'Inclination Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='Inclination (°)',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"Inclination stats:")
print(f"   Mean:  {df['inclination'].mean():.4f}°")
print(f"   Std:   {df['inclination'].std():.4f}°")
print(f"   Min:   {df['inclination'].min():.4f}°")
print(f"   Max:   {df['inclination'].max():.4f}°")
print(f"   Range: {df['inclination'].max() - df['inclination'].min():.4f}°")

Inclination stats:
   Mean:  97.3945°
   Std:   0.0496°
   Min:   97.3446°
   Max:   97.6284°
   Range: 0.2838°


In [6]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['eccentricity'],
    nbinsx=20,
    marker_color='magenta',
    marker_line=dict(color='black', width=1),
    name='Eccentricity'
))

fig.update_layout(
    title=f'Eccentricity Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='Eccentricity',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"Eccentricity stats:")
print(f"   Mean:  {df['eccentricity'].mean():.6f}")
print(f"   Std:   {df['eccentricity'].std():.6f}")
print(f"   Min:   {df['eccentricity'].min():.6f}")
print(f"   Max:   {df['eccentricity'].max():.6f}")
print(f"   Range: {(df['eccentricity'].max() - df['eccentricity'].min()):.6f}")
print(df.nlargest(3, 'eccentricity')[['name', 'intldes', 'norad', 'eccentricity']])

Eccentricity stats:
   Mean:  0.000866
   Std:   0.001575
   Min:   0.000178
   Max:   0.009959
   Range: 0.009781
              name     intldes  norad  eccentricity
28     ION SCV-013  2023-174AW  58300      0.009959
29  IMPULSE-1 MIRA  2023-174AX  58301      0.007411
11         TIGER-5   2023-174X  58277      0.001414


In [7]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['raan'],
    nbinsx=20,
    marker_color='orange',
    marker_line=dict(color='black', width=1),
    name='RAAN'
))

fig.update_layout(
    title=f'RAAN Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='RAAN (°)',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"RAAN stats:")
print(f"   Mean:  {df['raan'].mean():.4f}°")
print(f"   Std:   {df['raan'].std():.4f}°")
print(f"   Min:   {df['raan'].min():.4f}°")
print(f"   Max:   {df['raan'].max():.4f}°")
print(f"   Range: {df['raan'].max() - df['raan'].min():.4f}°")

RAAN stats:
   Mean:  197.8329°
   Std:   9.9772°
   Min:   163.0083°
   Max:   210.3630°
   Range: 47.3547°


In [8]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['arg_perigee'],
    nbinsx=20,
    marker_color='lime',
    marker_line=dict(color='black', width=1),
    name='Argument of Perigee'
))

fig.update_layout(
    title=f'Argument of Perigee Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='Argument of Perigee (°)',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"Argument of Perigee stats:")
print(f"   Mean:  {df['arg_perigee'].mean():.4f}°")
print(f"   Std:   {df['arg_perigee'].std():.4f}°")
print(f"   Min:   {df['arg_perigee'].min():.4f}°")
print(f"   Max:   {df['arg_perigee'].max():.4f}°")
print(f"   Range: {df['arg_perigee'].max() - df['arg_perigee'].min():.4f}°")

Argument of Perigee stats:
   Mean:  197.2313°
   Std:   142.1760°
   Min:   0.0140°
   Max:   359.9085°
   Range: 359.8945°


In [9]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['bstar'],
    nbinsx=20,
    marker_color='red',
    marker_line=dict(color='black', width=1),
    name='BSTAR'
))

fig.update_layout(
    title=f'BSTAR Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='BSTAR (1/earth radii)',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"BSTAR stats:")
print(f"   Mean:  {df['bstar'].mean():.6f}")
print(f"   Std:   {df['bstar'].std():.6f}")
print(f"   Min:   {df['bstar'].min():.6f}")
print(f"   Max:   {df['bstar'].max():.6f}")
print(f"   Range: {df['bstar'].max() - df['bstar'].min():.6f}")
print(df.nlargest(5, 'bstar')[['name', 'norad', 'bstar']])
print(df.nsmallest(5, 'bstar')[['name', 'norad', 'bstar']])

BSTAR stats:
   Mean:  0.000595
   Std:   0.000449
   Min:   0.000005
   Max:   0.001934
   Range: 0.001929
              name  norad     bstar
15     FLOCK 4Q 17  58283  0.001934
37     FLOCK 4Q 20  58313  0.001608
40     FLOCK 4Q 15  58319  0.001518
49          BRO-10  58331  0.001496
24  PELICAN-1 3001  58296  0.001495
            name  norad     bstar
23  SPACEVAN-001  58295  0.000005
42   PROTOMETHEE  58321  0.000093
52  AAC-HSI-SAT3  58848  0.000122
28   ION SCV-013  58300  0.000122
26          SPIP  58298  0.000155


In [14]:
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['mean_motion'],
    nbinsx=20,
    marker_color='yellow',
    marker_line=dict(color='black', width=1),
    name='Mean Motion'
))

fig.update_layout(
    title=f'Mean Motion Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='Mean Motion (revs/day)',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"Mean Motion stats:")
print(f"   Mean:  {df['mean_motion'].mean():.4f} rev/day")
print(f"   Std:   {df['mean_motion'].std():.4f} rev/day")
print(f"   Min:   {df['mean_motion'].min():.4f} rev/day")
print(f"   Max:   {df['mean_motion'].max():.4f} rev/day")
print(f"   Range: {df['mean_motion'].max() - df['mean_motion'].min():.4f} rev/day")
print(df.nlargest(5, 'mean_motion')[['name', 'norad', 'mean_motion']])

Mean Motion stats:
   Mean:  15.5117 rev/day
   Std:   0.2439 rev/day
   Min:   14.8440 rev/day
   Max:   16.4170 rev/day
   Range: 1.5729 rev/day
           name  norad  mean_motion
11      TIGER-5  58277    16.416971
49       BRO-10  58331    16.109902
15  FLOCK 4Q 17  58283    15.971778
6       TIGER-6  58272    15.814453
37  FLOCK 4Q 20  58313    15.752732


In [11]:
# Compute epoch age for each satellite
now = datetime.now(timezone.utc)

def parse_epoch(epoch_str):
    year = int(epoch_str[:2])
    day  = float(epoch_str[2:])
    full_year = 2000 + year if year < 57 else 1900 + year
    from datetime import timedelta
    base = datetime(full_year, 1, 1, tzinfo=timezone.utc)
    return base + timedelta(days=day - 1)

df['epoch_dt']  = df['epoch'].apply(parse_epoch)
df['epoch_age_hrs'] = [(now - e).total_seconds() / 3600 for e in df['epoch_dt']]

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df['epoch_age_hrs'],
    nbinsx=20,
    marker_color='lightblue',
    marker_line=dict(color='black', width=1),
    name='Epoch Age'
))

fig.update_layout(
    title=f'Epoch Age Distribution — {satcat_cache["intldes"]} ({len(df)} satellites)',
    xaxis_title='Epoch Age (hours)',
    yaxis_title='Number of Satellites',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"Epoch Age stats:")
print(f"   Mean:  {df['epoch_age_hrs'].mean():.1f} hrs")
print(f"   Std:   {df['epoch_age_hrs'].std():.1f} hrs")
print(f"   Min:   {df['epoch_age_hrs'].min():.1f} hrs")
print(f"   Max:   {df['epoch_age_hrs'].max():.1f} hrs")

# Flag stale TLEs
stale = df[df['epoch_age_hrs'] > 48]
print(f"\nStale TLEs (> 48hrs): {len(stale)}")
for _, row in stale.iterrows():
    print(f"   {row['name']:<25} {row['epoch_age_hrs']:.1f} hrs old")

Epoch Age stats:
   Mean:  10.9 hrs
   Std:   6.2 hrs
   Min:   5.1 hrs
   Max:   34.9 hrs

Stale TLEs (> 48hrs): 0


# Section 2 - Propagated Dynamics 

In [12]:
# Section 2 — Propagated Dynamics
# Item 8 — Altitude Summary Table

alt_df = df[['name', 'norad', 'bstar', 'alt_min', 'alt_max', 'alt_mean']].copy()
alt_df['alt_range'] = alt_df['alt_max'] - alt_df['alt_min']
alt_df = alt_df.sort_values('alt_mean', ascending=False).reset_index(drop=True)

print(f"{'Name':<25} {'NORAD':<8} {'Min Alt':>8} {'Max Alt':>8} {'Mean Alt':>9} {'Range':>8}")
print("-" * 75)
for _, row in alt_df.iterrows():
    print(f"{row['name']:<25} {row['norad']:<8} {row['alt_min']:>8.1f} {row['alt_max']:>8.1f} {row['alt_mean']:>9.1f} {row['alt_range']:>8.1f}")

print(f"\nFleet-wide stats:")
print(f"   Highest mean alt:  {alt_df['alt_mean'].max():.1f} km — {alt_df.iloc[0]['name']}")
print(f"   Lowest mean alt:   {alt_df['alt_mean'].min():.1f} km — {alt_df.iloc[-1]['name']}")
print(f"   Fleet mean alt:    {alt_df['alt_mean'].mean():.1f} km")
print(f"   Fleet alt spread:  {alt_df['alt_mean'].max() - alt_df['alt_mean'].min():.1f} km")

Name                      NORAD     Min Alt  Max Alt  Mean Alt    Range
---------------------------------------------------------------------------
AAC-AIS-SAT3              58997       612.9    648.0     627.3     35.0
ION SCV-013               58300       485.6    634.8     564.0    149.2
UMBRA-07                  58297       544.0    569.3     554.0     25.3
AAC-HSI-SAT3              58848       515.2    547.1     531.5     31.9
BRO-11                    58264       481.1    520.3     497.6     39.2
FALCONSAT-10              58291       480.6    516.7     496.6     36.1
SPIP                      58298       480.0    516.8     496.4     36.8
PROTOMETHEE               58321       470.9    504.2     485.7     33.3
ICEYE-X31                 58288       468.4    498.4     480.8     30.0
GENMAT-1                  58276       462.6    496.7     477.4     34.1
EXO-0                     59175       460.7    495.7     476.7     35.0
IMPULSE-1 MIRA            58301       423.4    522.2     476

In [15]:
# Item 9 — Orbital Period per Satellite

df['period_min'] = 24 * 60 / df['mean_motion']

period_df = df[['name', 'norad', 'alt_mean', 'mean_motion', 'period_min']].copy()
period_df = period_df.sort_values('period_min', ascending=False).reset_index(drop=True)

# Plot
fig = go.Figure()

fig.add_trace(go.Bar(
    x=period_df['name'],
    y=period_df['period_min'],
    marker_color='cyan',
    marker_line=dict(color='black', width=1),
    name='Orbital Period'
))

fig.update_layout(
    title=f'Orbital Period per Satellite — {satcat_cache["intldes"]}',
    xaxis_title='Satellite',
    yaxis_title='Orbital Period (min)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray', tickangle=45),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"{'Name':<25} {'NORAD':<8} {'Mean Alt':>9} {'Mean Motion':>12} {'Period':>10}")
print("-" * 70)
for _, row in period_df.iterrows():
    print(f"{row['name']:<25} {row['norad']:<8} {row['alt_mean']:>9.1f} {row['mean_motion']:>12.4f} {row['period_min']:>10.2f}")

print(f"\nFleet-wide stats:")
print(f"   Longest period:  {period_df['period_min'].max():.2f} min — {period_df.iloc[0]['name']}")
print(f"   Shortest period: {period_df['period_min'].min():.2f} min — {period_df.iloc[-1]['name']}")
print(f"   Fleet mean:      {period_df['period_min'].mean():.2f} min")
print(f"   Range:           {period_df['period_min'].max() - period_df['period_min'].min():.2f} min")

Name                      NORAD     Mean Alt  Mean Motion     Period
----------------------------------------------------------------------
AAC-AIS-SAT3              58997        627.3      14.8440      97.01
ION SCV-013               58300        564.0      15.0488      95.69
UMBRA-07                  58297        554.0      15.0807      95.49
AAC-HSI-SAT3              58848        531.5      15.1549      95.02
BRO-11                    58264        497.6      15.2680      94.31
FALCONSAT-10              58291        496.6      15.2716      94.29
SPIP                      58298        496.4      15.2717      94.29
PROTOMETHEE               58321        485.7      15.3065      94.08
ICEYE-X31                 58288        480.8      15.3243      93.97
GENMAT-1                  58276        477.4      15.3359      93.90
EXO-0                     59175        476.7      15.3370      93.89
AETHER-2                  58299        476.2      15.3388      93.88
IMPULSE-1 MIRA            58301 

# Altitude Decay Rate 15-18

In [16]:
from numpy.polynomial import polynomial as P

# Item 10 — Altitude Decay Rate
times_hrs = np.linspace(0, 24, r_teme.shape[1])

decay_rates = []
for i in range(len(tles)):
    valid_mask = e[i] == 0
    if valid_mask.sum() < 10:
        decay_rates.append(np.nan)
        continue
    # Linear fit on altitude over time
    t_valid   = times_hrs[valid_mask]
    alt_valid = alts[i][valid_mask]
    coeffs    = np.polyfit(t_valid, alt_valid, 1)  # slope, intercept
    decay_rates.append(coeffs[0] * 24)  # km/day

df['decay_rate_km_day'] = decay_rates

decay_df = df[['name', 'norad', 'bstar', 'alt_mean', 'decay_rate_km_day']].copy()
decay_df = decay_df.sort_values('decay_rate_km_day').reset_index(drop=True)

# Plot
fig = go.Figure()

fig.add_trace(go.Bar(
    x=decay_df['name'],
    y=decay_df['decay_rate_km_day'],
    marker_color='red',
    marker_line=dict(color='black', width=1),
    name='Altitude Decay Rate'
))

fig.update_layout(
    title=f'Altitude Decay Rate per Satellite — {satcat_cache["intldes"]}',
    xaxis_title='Satellite',
    yaxis_title='Decay Rate (km/day)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray', tickangle=45),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

print(f"{'Name':<25} {'NORAD':<8} {'BSTAR':>10} {'Mean Alt':>9} {'Decay km/day':>13}")
print("-" * 70)
for _, row in decay_df.iterrows():
    print(f"{row['name']:<25} {row['norad']:<8} {row['bstar']:>10.6f} {row['alt_mean']:>9.1f} {row['decay_rate_km_day']:>13.4f}")

print(f"\nFleet-wide stats:")
print(f"   Fastest decay:  {decay_df['decay_rate_km_day'].min():.4f} km/day — {decay_df.iloc[0]['name']}")
print(f"   Slowest decay:  {decay_df['decay_rate_km_day'].max():.4f} km/day — {decay_df.iloc[-1]['name']}")
print(f"   Fleet mean:     {decay_df['decay_rate_km_day'].mean():.4f} km/day")

Name                      NORAD         BSTAR  Mean Alt  Decay km/day
----------------------------------------------------------------------
TIGER-5                   58277      0.000473     110.3      -54.1185
BRO-10                    58331      0.001496     249.4       -9.4548
FLOCK 4Q 17               58283      0.001934     289.8       -5.4271
ION SCV-013               58300      0.000122     564.0       -3.1276
FLOCK 4Q 15               58319      0.001518     356.6       -1.2902
AAC-HSI-SAT3              58848      0.000122     531.5       -1.2429
FLOCK 4Q 20               58313      0.001608     354.7       -1.2352
FLOCK 4Q 12               58275      0.001124     399.2       -1.1427
TIGER-6                   58272      0.001056     337.3       -1.0120
FLOCK 4Q 21               58329      0.001308     380.6       -1.0016
EXO-0                     59175      0.000267     476.7       -0.9768
PELICAN-1 3001            58296      0.001495     371.1       -0.8427
AETHER-2           

In [17]:
# Rank by BSTAR and decay rate separately
df['bstar_rank']      = df['bstar'].rank(ascending=False)
df['decay_rank']      = df['decay_rate_km_day'].rank(ascending=True)
df['rank_difference'] = (df['bstar_rank'] - df['decay_rank']).abs()

compare_df = df[['name', 'norad', 'bstar', 'bstar_rank', 
                  'decay_rate_km_day', 'decay_rank', 
                  'rank_difference', 'alt_mean']].copy()
compare_df = compare_df.sort_values('rank_difference', ascending=False)

print(f"{'Name':<25} {'BSTAR Rank':>10} {'Decay Rank':>10} {'Rank Diff':>10} {'Mean Alt':>9}")
print("-" * 70)
for _, row in compare_df.iterrows():
    print(f"{row['name']:<25} {row['bstar_rank']:>10.0f} {row['decay_rank']:>10.0f} "
          f"{row['rank_difference']:>10.0f} {row['alt_mean']:>9.1f}")

# Correlation coefficient
corr = df['bstar'].corr(df['decay_rate_km_day'])
print(f"\nPearson correlation BSTAR vs decay rate: {corr:.4f}")

# Partial correlation — controlling for altitude
from numpy import corrcoef
alt_controlled = df['decay_rate_km_day'] / np.exp(-df['alt_mean'] / 8.5)
corr_alt = df['bstar'].corr(pd.Series(alt_controlled))
print(f"Altitude-controlled correlation:         {corr_alt:.4f}")

Name                      BSTAR Rank Decay Rank  Rank Diff  Mean Alt
----------------------------------------------------------------------
ION SCV-013                       52          4         48     564.0
AAC-HSI-SAT3                      53          6         47     531.5
AETHER-2                          50         13         37     476.2
EXO-0                             46         11         35     476.7
UMBRA-07                          21         52         31     554.0
SPACEVAN-001                      55         25         30     407.9
BRO-11                            49         22         27     497.6
FLOCK 4Q 9                        23         46         23     437.2
TIGER-5                           24          1         23     110.3
ORBASTRO-PC-1                     28         50         22     439.8
FLOCK 4Q 34                        9         29         20     397.7
FLOCK 4Q 23                        8         28         20     400.2
FLOCK 4Q 4                      

In [21]:
# Remove known active maneuverers
exclude = ['ION SCV-013', 'IMPULSE-1 MIRA', 'TIGER-5', 'AAC-AIS-SAT3', 'AAC-HSI-SAT3']

df_clean = df[~df['name'].isin(exclude)].copy()
df_clean['rho_relative']     = np.exp(-df_clean['alt_mean'] / H)
df_clean['normalized_decay'] = df_clean['decay_rate_km_day'] / df_clean['rho_relative']

corr_raw        = df_clean['bstar'].corr(df_clean['decay_rate_km_day'])
corr_normalized = df_clean['bstar'].corr(df_clean['normalized_decay'])

print(f"Clean dataset — {len(df_clean)} satellites (excluded {len(exclude)} maneuverers)")
print(f"Raw correlation:              {corr_raw:.4f}")
print(f"Altitude-normalized corr:     {corr_normalized:.4f}")

# Normalize decay rate by atmospheric density using exponential scale height model
# ρ(h) = ρ₀ × exp(-h / H)
# H = 8.5 km scale height (approximate for LEO)
# We just need relative density so ρ₀ cancels out

H = 8.5  # km scale height

df['rho_relative']       = np.exp(-df['alt_mean'] / H)
df['normalized_decay']   = df['decay_rate_km_day'] / df['rho_relative']

corr_normalized = df['bstar'].corr(df['normalized_decay'])
print(f"Raw Pearson correlation:        {df['bstar'].corr(df['decay_rate_km_day']):.4f}")
print(f"Altitude-normalized correlation: {corr_normalized:.4f}")

# Plot BSTAR vs normalized decay
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['bstar'],
    y=df['normalized_decay'],
    mode='markers+text',
    text=df['name'],
    textposition='top center',
    textfont=dict(size=8),
    marker=dict(size=8, color='cyan'),
))

fig.update_layout(
    title='BSTAR vs Altitude-Normalized Decay Rate',
    xaxis_title='BSTAR',
    yaxis_title='Normalized Decay Rate',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()

Clean dataset — 50 satellites (excluded 5 maneuverers)
Raw correlation:              -0.5866
Altitude-normalized corr:     -0.0445
Raw Pearson correlation:        -0.0765
Altitude-normalized correlation: -0.0971


In [23]:
# Install if needed
# pip install nrlmsise00 pyatmos --break-system-packages

from nrlmsise00 import msise_flat
from datetime import datetime, timezone
import pandas as pd

# Reference datetime for atmospheric conditions
ref_time = datetime.now(timezone.utc)

def get_nrlmsise_density(alt_km, lat=0, lon=0, dt=ref_time):
    """Get atmospheric density from NRLMSISE-00 at given altitude"""
    result = msise_flat(dt, alt_km, lat, lon, 150, 150, 4)
    # Total mass density in kg/m³ — index 5
    return result[5]

# Compute NRLMSISE-00 density at mean altitude for each satellite
print("Computing NRLMSISE-00 densities...")
df_clean['rho_nrlmsise'] = df_clean['alt_mean'].apply(
    lambda h: get_nrlmsise_density(h)
)

# Normalized decay using NRLMSISE-00
df_clean['normalized_decay_nrl'] = df_clean['decay_rate_km_day'] / df_clean['rho_nrlmsise']

corr_nrl = df_clean['bstar'].corr(df_clean['normalized_decay_nrl'])
print(f"NRLMSISE-00 normalized correlation: {corr_nrl:.4f}")

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_clean['bstar'],
    y=df_clean['normalized_decay_nrl'],
    mode='markers+text',
    text=df_clean['name'],
    textposition='top center',
    textfont=dict(size=8),
    marker=dict(size=8, color='cyan'),
    name='NRLMSISE-00'
))

fig.update_layout(
    title='BSTAR vs NRLMSISE-00 Normalized Decay Rate',
    xaxis_title='BSTAR',
    yaxis_title='Normalized Decay Rate',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)

fig.show()
print(f"Summary:")
print(f"   Raw correlation:               {df_clean['bstar'].corr(df_clean['decay_rate_km_day']):.4f}")
print(f"   Simple exp normalized:         -0.0445")
print(f"   NRLMSISE-00 normalized:        {corr_nrl:.4f}")

Computing NRLMSISE-00 densities...
NRLMSISE-00 normalized correlation: -0.1396


Summary:
   Raw correlation:               -0.5866
   Simple exp normalized:         -0.0445
   NRLMSISE-00 normalized:        -0.1396


# Ground Track Coverage 

In [26]:
# Item 11 — Ground Track Coverage
lat_coverage = []
for i in range(len(tles)):
    valid = lats[i][e[i] == 0]
    lat_coverage.append({
        'name':    tles[i]['name'],
        'lat_min': valid.min(),
        'lat_max': valid.max(),
        'lat_range': valid.max() - valid.min()
    })

cov_df = pd.DataFrame(lat_coverage).sort_values('lat_range', ascending=False)
print(f"{'Name':<25} {'Lat Min':>8} {'Lat Max':>8} {'Range':>8}")
print("-" * 55)
for _, row in cov_df.iterrows():
    print(f"{row['name']:<25} {row['lat_min']:>8.1f} {row['lat_max']:>8.1f} {row['lat_range']:>8.1f}")

Name                       Lat Min  Lat Max    Range
-------------------------------------------------------
TIGER-5                      -82.7     82.7    165.4
AAC-HSI-SAT3                 -82.7     82.7    165.4
BRO-10                       -82.7     82.7    165.4
FLOCK 4Q 17                  -82.7     82.7    165.3
GENMAT-1                     -82.7     82.7    165.3
ORBASTRO-PC-1                -82.7     82.7    165.3
PROTOMETHEE                  -82.7     82.7    165.3
PICO-01B009                  -82.7     82.7    165.3
ICEYE-X31                    -82.7     82.7    165.3
PICO-01A003                  -82.7     82.7    165.3
ICEYE-X35                    -82.7     82.7    165.3
FLOCK 4Q 1                   -82.7     82.7    165.3
FLOCK 4Q 22                  -82.7     82.7    165.3
FLOCK 4Q 25                  -82.7     82.7    165.3
FLOCK 4Q 27                  -82.7     82.7    165.3
FLOCK 4Q 34                  -82.7     82.7    165.3
FLOCK 4Q 4                   -82.7     82.7

# Object Separation Distance

In [28]:
# Item 12 — Object Separation Distance Over Time
# Compute pairwise distances between all satellites at each timestep
# Use TEME positions directly — already in km
num_steps = r_teme.shape[1]
# Pick a subset of timesteps for efficiency (every 10th point = 200 points)
step = 10
t_subset = np.arange(0, num_steps, step)
times_hrs_subset = np.linspace(0, 24, len(t_subset))

# Compute mean pairwise separation at each timestep
n_sats = len(tles)
mean_separations = []
max_separations  = []
min_separations  = []

for j in t_subset:
    positions = r_teme[:, j, :]  # (n_sats, 3)
    dists = []
    for a in range(n_sats):
        for b in range(a+1, n_sats):
            if e[a, j] == 0 and e[b, j] == 0:
                d = np.linalg.norm(positions[a] - positions[b])
                dists.append(d)
    if dists:
        mean_separations.append(np.mean(dists))
        max_separations.append(np.max(dists))
        min_separations.append(np.min(dists))
    else:
        mean_separations.append(np.nan)
        max_separations.append(np.nan)
        min_separations.append(np.nan)

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=times_hrs_subset,
    y=mean_separations,
    mode='lines',
    name='Mean Separation',
    line=dict(color='cyan', width=2)
))

fig.add_trace(go.Scatter(
    x=times_hrs_subset,
    y=max_separations,
    mode='lines',
    name='Max Separation',
    line=dict(color='red', width=1, dash='dash')
))

fig.add_trace(go.Scatter(
    x=times_hrs_subset,
    y=min_separations,
    mode='lines',
    name='Min Separation',
    line=dict(color='lime', width=1, dash='dash')
))

fig.update_layout(
    title=f'Object Separation Distance Over 24hrs — {satcat_cache["intldes"]}',
    xaxis_title='Time (hrs)',
    yaxis_title='Separation Distance (km)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)

fig.show()

print(f"Separation stats:")
print(f"   Initial mean sep:  {mean_separations[0]:.1f} km")
print(f"   Final mean sep:    {mean_separations[-1]:.1f} km")
print(f"   Initial min sep:   {min_separations[0]:.1f} km")
print(f"   Final min sep:     {min_separations[-1]:.1f} km")
print(f"   Initial max sep:   {max_separations[0]:.1f} km")
print(f"   Final max sep:     {max_separations[-1]:.1f} km")

Separation stats:
   Initial mean sep:  8676.0 km
   Final mean sep:    8607.7 km
   Initial min sep:   110.4 km
   Final min sep:     117.9 km
   Initial max sep:   13717.3 km
   Final max sep:     13744.0 km


# Object Separation on Historical Data

In [30]:
# Load historical TLE cache
HISTORY_CACHE_FILE = os.path.expanduser("~/tle_history_cache.json")

with open(HISTORY_CACHE_FILE) as f:
    history_cache = json.load(f)

historical_tles = history_cache['tles']
hist_df = pd.DataFrame(historical_tles)
hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
hist_df['is_confirmed'] = hist_df['name'].apply(
    lambda n: not (n.startswith('TBA') or n.startswith('OBJECT'))
)

print(f"Historical TLE cache loaded!")
print(f"   Launch:      {history_cache['intldes']}")
print(f"   Epoch range: {history_cache['start_date']} → {history_cache['end_date']}")
print(f"   Records:     {len(hist_df)}")
print(f"   Unique sats: {hist_df['norad'].nunique()}")

Historical TLE cache loaded!
   Launch:      2023-174
   Epoch range: 2023-11-11 → 2024-01-11
   Records:     10185
   Unique sats: 92


In [39]:
# Get first TLE per satellite
first_tles = (hist_df.sort_values('epoch')
                     .groupby('norad')
                     .first()
                     .reset_index())

# Get TLE closest to 7 days after first TLE
def get_7day_tle(norad, first_epoch):
    target = first_epoch + pd.Timedelta(days=2)
    sat_tles = hist_df[hist_df['norad'] == norad].copy()
    sat_tles['time_diff'] = (sat_tles['epoch'] - target).abs()
    return sat_tles.nsmallest(1, 'time_diff').iloc[0]

rows = []
for _, row in first_tles.iterrows():
    later = get_7day_tle(row['norad'], row['epoch'])
    rows.append({
        'norad':         row['norad'],
        'name':          row['name'],
        'epoch_t0':      row['epoch'],
        'line1_t0':      row['line1'],
        'line2_t0':      row['line2'],
        'epoch_t7':      later['epoch'],
        'line1_t7':      later['line1'],
        'line2_t7':      later['line2'],
        'actual_dt_days':(later['epoch'] - row['epoch']).total_seconds() / 86400
    })

pair_df = pd.DataFrame(rows)
pair_df = pair_df[pair_df['line1_t0'].notna() & pair_df['line1_t7'].notna()]

print(f"Satellite pairs: {len(pair_df)}")
print(f"Mean actual dt:  {pair_df['actual_dt_days'].mean():.2f} days")
print(f"Min actual dt:   {pair_df['actual_dt_days'].min():.2f} days")
print(f"Max actual dt:   {pair_df['actual_dt_days'].max():.2f} days")

Satellite pairs: 92
Mean actual dt:  2.03 days
Min actual dt:   0.00 days
Max actual dt:   2.51 days


In [40]:
from sgp4.api import Satrec

errors_km = []

for _, row in pair_df.iterrows():
    try:
        # Parse both TLEs
        sat_t0 = Satrec.twoline2rv(row['line1_t0'], row['line2_t0'])
        sat_t7 = Satrec.twoline2rv(row['line1_t7'], row['line2_t7'])

        # Evaluate both at the t7 epoch — same moment in time
        eval_time = Time(row['epoch_t7'])
        jd1, jd2  = eval_time.jd1, eval_time.jd2

        # Propagate t0 TLE forward to t7 epoch
        e0, r0, v0 = sat_t0.sgp4(jd1, jd2)

        # Propagate t7 TLE at its own epoch (reference position)
        e7, r7, v7 = sat_t7.sgp4(jd1, jd2)

        if e0 != 0 or e7 != 0:
            errors_km.append(np.nan)
            continue

        # Position error
        error = np.linalg.norm(np.array(r0) - np.array(r7))
        errors_km.append(error)

    except Exception as ex:
        errors_km.append(np.nan)

pair_df['position_error_km'] = errors_km

print(f"{'Name':<25} {'NORAD':<8} {'dt (days)':>10} {'Error (km)':>12}")
print("-" * 60)
for _, row in pair_df.sort_values('position_error_km', ascending=False).iterrows():
    print(f"{row['name']:<25} {row['norad']:<8} {row['actual_dt_days']:>10.2f} {row['position_error_km']:>12.2f}")

print(f"\nPosition error stats:")
print(f"   Mean:    {pair_df['position_error_km'].mean():.2f} km")
print(f"   Median:  {pair_df['position_error_km'].median():.2f} km")
print(f"   Min:     {pair_df['position_error_km'].min():.2f} km")
print(f"   Max:     {pair_df['position_error_km'].max():.2f} km")
print(f"   Std:     {pair_df['position_error_km'].std():.2f} km")

Name                      NORAD     dt (days)   Error (km)
------------------------------------------------------------
TBA - TO BE ASSIGNED      58566          1.78        20.34
TBA - TO BE ASSIGNED      58345          2.24        16.36
TBA - TO BE ASSIGNED      58343          2.11        14.77
OBJECT AT                 58297          2.11        12.83
TBA - TO BE ASSIGNED      58337          2.11        12.82
OBJECT M                  58267          2.12        11.94
TBA - TO BE ASSIGNED      58564          2.04        11.55
TBA - TO BE ASSIGNED      58344          2.11        11.43
OBJECT AN                 58292          2.18        11.31
TBA - TO BE ASSIGNED      58308          2.51         9.88
OBJECT BG                 58310          2.11         8.83
TBA - TO BE ASSIGNED      58270          2.11         7.39
TBA - TO BE ASSIGNED      58563          2.04         6.94
TBA - TO BE ASSIGNED      58569          1.98         6.12
OBJECT AB                 58281          2.11         

In [41]:
# Error vs time — propagate t0 TLE at multiple intervals
time_points = [0.5, 1, 2, 3, 5, 7, 10, 14, 21, 30]  # days

def get_nearest_tle(norad, first_epoch, target_days):
    target = first_epoch + pd.Timedelta(days=target_days)
    sat_tles = hist_df[hist_df['norad'] == norad].copy()
    sat_tles['time_diff'] = (sat_tles['epoch'] - target).abs()
    nearest = sat_tles.nsmallest(1, 'time_diff').iloc[0]
    actual_dt = (nearest['epoch'] - first_epoch).total_seconds() / 86400
    return nearest, actual_dt

results = []
for target_days in time_points:
    errors = []
    for _, row in pair_df.iterrows():
        try:
            nearest, actual_dt = get_nearest_tle(row['norad'], row['epoch_t0'], target_days)
            if pd.isna(nearest['line1']) or pd.isna(nearest['line2']):
                continue

            sat_t0 = Satrec.twoline2rv(row['line1_t0'], row['line2_t0'])
            sat_tn = Satrec.twoline2rv(nearest['line1'], nearest['line2'])

            eval_time = Time(nearest['epoch'])
            jd1, jd2  = eval_time.jd1, eval_time.jd2

            e0, r0, _ = sat_t0.sgp4(jd1, jd2)
            en, rn, _ = sat_tn.sgp4(jd1, jd2)

            if e0 != 0 or en != 0:
                continue

            error = np.linalg.norm(np.array(r0) - np.array(rn))
            errors.append(error)
        except:
            continue

    results.append({
        'days':   target_days,
        'mean':   np.nanmean(errors),
        'median': np.nanmedian(errors),
        'p75':    np.nanpercentile(errors, 75),
        'p90':    np.nanpercentile(errors, 90),
        'min':    np.nanmin(errors)
    })

results_df = pd.DataFrame(results)

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results_df['days'], y=results_df['median'],
    mode='lines+markers', name='Median Error',
    line=dict(color='cyan', width=2)
))
fig.add_trace(go.Scatter(
    x=results_df['days'], y=results_df['mean'],
    mode='lines+markers', name='Mean Error',
    line=dict(color='orange', width=2)
))
fig.add_trace(go.Scatter(
    x=results_df['days'], y=results_df['p75'],
    mode='lines+markers', name='75th Percentile',
    line=dict(color='yellow', width=1, dash='dash')
))
fig.add_trace(go.Scatter(
    x=results_df['days'], y=results_df['p90'],
    mode='lines+markers', name='90th Percentile',
    line=dict(color='red', width=1, dash='dash')
))

# Recommended refresh zones
fig.add_vrect(x0=0, x1=2, fillcolor='green', opacity=0.1,
              annotation_text='Refresh Zone', annotation_position='top left')
fig.add_vrect(x0=2, x1=7, fillcolor='yellow', opacity=0.1,
              annotation_text='Caution Zone', annotation_position='top left')
fig.add_vrect(x0=7, x1=30, fillcolor='red', opacity=0.1,
              annotation_text='Stale Zone', annotation_position='top left')

fig.update_layout(
    title='SGP4 Propagation Error vs Time — Preliminary TLEs',
    xaxis_title='Days Since First TLE',
    yaxis_title='Position Error (km)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)

fig.show()

print(f"\n{'Days':>6} {'Median':>10} {'Mean':>10} {'P75':>10} {'P90':>10}")
print("-" * 50)
for _, row in results_df.iterrows():
    print(f"{row['days']:>6.1f} {row['median']:>10.2f} {row['mean']:>10.2f} {row['p75']:>10.2f} {row['p90']:>10.2f}")


  Days     Median       Mean        P75        P90
--------------------------------------------------
   0.5       0.00       0.08       0.00       0.00
   1.0       0.93       1.35       1.61       2.63
   2.0       2.07       3.44       3.81       9.77
   3.0       4.20       9.15       7.92      23.65
   5.0      14.34      33.99      30.21      78.74
   7.0      21.07      62.33      45.51     123.66
  10.0      16.61      94.82      58.01     242.72
  14.0     111.55     333.32     184.12     678.39
  21.0     440.09    1019.23     776.80    2265.71
  30.0    1003.02    1903.59    1795.84    4893.36


In [49]:
prop = np.load(PROP_CACHE_FILE)
r_teme = prop['r_teme']
v_teme = prop['v_teme']
e      = prop['e']
lats   = prop['lats']
lons   = prop['lons']
alts   = prop['alts']
times  = Time(prop['times_jd1'], prop['times_jd2'], format='jd', scale='utc')

print(f"Propagation cache loaded!")
print(f"   Shape: {r_teme.shape}")

Propagation cache loaded!
   Shape: (55, 2000, 3)


In [50]:
# Cluster satellites by mean altitude
from sklearn.cluster import KMeans
import plotly.express as px

# Get mean altitude per satellite from current TLEs
alt_array = np.array([alts[i][e[i]==0].mean() for i in range(len(tles))])
names_array = np.array([t['name'] for t in tles])

# Plot altitude distribution to visually identify clusters
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=alt_array,
    nbinsx=30,
    marker_color='cyan',
    marker_line=dict(color='black', width=1)
))

fig.update_layout(
    title='Altitude Distribution — Natural Clusters',
    xaxis_title='Mean Altitude (km)',
    yaxis_title='Count',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
)
fig.show()

# Print altitude sorted
alt_df = pd.DataFrame({
    'name':     names_array,
    'alt_mean': alt_array
}).sort_values('alt_mean')

print(f"\n{'Name':<25} {'Mean Alt (km)':>14}")
print("-" * 42)
for _, row in alt_df.iterrows():
    print(f"{row['name']:<25} {row['alt_mean']:>14.1f}")


Name                       Mean Alt (km)
------------------------------------------
TIGER-5                            110.3
BRO-10                             249.4
FLOCK 4Q 17                        289.8
TIGER-6                            337.3
FLOCK 4Q 20                        354.7
FLOCK 4Q 15                        356.6
FLOCK 4Q 1                         368.4
PELICAN-1 3001                     371.1
PICO-01B009                        372.8
AETHER-1                           377.9
FLOCK 4Q 21                        380.6
FLOCK 4Q 25                        383.6
FLOCK 4Q 34                        397.7
FLOCK 4Q 12                        399.2
FLOCK 4Q 23                        400.2
PLATERO                            400.5
ICEYE-X34                          407.4
SPACEVAN-001                       407.9
FLOCK 4Q 22                        412.4
FLOCK 4Q 33                        416.5
FLOCK 4Q 36                        416.9
FLOCK 4Q 30                        417.3
FLOCK 4Q 26  

In [51]:
# Simple altitude-based clustering
alt_df = pd.DataFrame({
    'name':     [t['name'] for t in tles],
    'alt_mean': [alts[i][e[i]==0].mean() for i in range(len(tles))]
}).sort_values('alt_mean').reset_index(drop=True)

# Define clusters based on natural altitude breaks
def assign_cluster(alt):
    if alt < 300:
        return 'Low — Decaying (<300km)'
    elif alt < 520:
        return 'Core Cluster (300-520km)'
    else:
        return 'High Shell (>520km)'

alt_df['cluster'] = alt_df['alt_mean'].apply(assign_cluster)

# Plot
fig = go.Figure()

colors = {
    'Low — Decaying (<300km)':  'red',
    'Core Cluster (300-520km)': 'cyan',
    'High Shell (>520km)':      'orange'
}

for cluster, group in alt_df.groupby('cluster'):
    fig.add_trace(go.Scatter(
        x=group['alt_mean'],
        y=[cluster] * len(group),
        mode='markers+text',
        text=group['name'],
        textposition='top center',
        textfont=dict(size=8),
        marker=dict(size=10, color=colors[cluster]),
        name=cluster
    ))

fig.update_layout(
    title=f'Altitude Clusters — {satcat_cache["intldes"]}',
    xaxis_title='Mean Altitude (km)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)
fig.show()

# Summary
for cluster, group in alt_df.groupby('cluster'):
    print(f"\n{cluster} — {len(group)} satellites")
    print(f"   Alt range: {group['alt_mean'].min():.1f} — {group['alt_mean'].max():.1f} km")
    for _, row in group.iterrows():
        print(f"   {row['name']:<25} {row['alt_mean']:.1f} km")


Core Cluster (300-520km) — 48 satellites
   Alt range: 337.3 — 497.6 km
   TIGER-6                   337.3 km
   FLOCK 4Q 20               354.7 km
   FLOCK 4Q 15               356.6 km
   FLOCK 4Q 1                368.4 km
   PELICAN-1 3001            371.1 km
   PICO-01B009               372.8 km
   AETHER-1                  377.9 km
   FLOCK 4Q 21               380.6 km
   FLOCK 4Q 25               383.6 km
   FLOCK 4Q 34               397.7 km
   FLOCK 4Q 12               399.2 km
   FLOCK 4Q 23               400.2 km
   PLATERO                   400.5 km
   ICEYE-X34                 407.4 km
   SPACEVAN-001              407.9 km
   FLOCK 4Q 22               412.4 km
   FLOCK 4Q 33               416.5 km
   FLOCK 4Q 36               416.9 km
   FLOCK 4Q 30               417.3 km
   FLOCK 4Q 26               417.9 km
   FLOCK 4Q 31               418.1 km
   FLOCK 4Q 35               419.4 km
   FLOCK 4Q 29               421.7 km
   FLOCK 4Q 7                422.7 km
   FLOCK 4Q 28 

In [53]:
def assign_cluster(alt):
    if alt < 300:
        return 'A — Decaying (<300km)'
    elif alt < 400:
        return 'B — Lower Core (300-400km)'
    elif alt < 480:
        return 'C — Upper Core (400-480km)'
    else:
        return 'D — High Shell (>480km)'

alt_df['cluster'] = alt_df['alt_mean'].apply(assign_cluster)

colors = {
    'A — Decaying (<300km)':    'red',
    'B — Lower Core (300-400km)': 'orange',
    'C — Upper Core (400-480km)': 'cyan',
    'D — High Shell (>480km)':  'magenta'
}

fig = go.Figure()

for cluster, group in alt_df.groupby('cluster'):
    fig.add_trace(go.Scatter(
        x=group['alt_mean'],
        y=[cluster] * len(group),
        mode='markers+text',
        text=group['name'],
        textposition='top center',
        textfont=dict(size=8),
        marker=dict(size=10, color=colors[cluster]),
        name=cluster
    ))

fig.update_layout(
    title=f'Refined Altitude Clusters — {satcat_cache["intldes"]}',
    xaxis_title='Mean Altitude (km)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)
fig.show()

for cluster, group in alt_df.groupby('cluster'):
    print(f"\n{cluster} — {len(group)} satellites")
    print(f"   Alt range: {group['alt_mean'].min():.1f} — {group['alt_mean'].max():.1f} km")
    for _, row in group.iterrows():
        print(f"   {row['name']:<25} {row['alt_mean']:.1f} km")


A — Decaying (<300km) — 3 satellites
   Alt range: 110.3 — 289.8 km
   TIGER-5                   110.3 km
   BRO-10                    249.4 km
   FLOCK 4Q 17               289.8 km

B — Lower Core (300-400km) — 11 satellites
   Alt range: 337.3 — 399.2 km
   TIGER-6                   337.3 km
   FLOCK 4Q 20               354.7 km
   FLOCK 4Q 15               356.6 km
   FLOCK 4Q 1                368.4 km
   PELICAN-1 3001            371.1 km
   PICO-01B009               372.8 km
   AETHER-1                  377.9 km
   FLOCK 4Q 21               380.6 km
   FLOCK 4Q 25               383.6 km
   FLOCK 4Q 34               397.7 km
   FLOCK 4Q 12               399.2 km

C — Upper Core (400-480km) — 32 satellites
   Alt range: 400.2 — 477.4 km
   FLOCK 4Q 23               400.2 km
   PLATERO                   400.5 km
   ICEYE-X34                 407.4 km
   SPACEVAN-001              407.9 km
   FLOCK 4Q 22               412.4 km
   FLOCK 4Q 33               416.5 km
   FLOCK 4Q 36       

In [54]:
# Filter to cluster C for separation analysis
cluster_c = alt_df[alt_df['cluster'] == 'C — Upper Core (400-480km)']
cluster_c_names = cluster_c['name'].tolist()
cluster_c_idx   = [i for i, t in enumerate(tles) if t['name'] in cluster_c_names]

print(f"Cluster C satellites: {len(cluster_c_idx)}")
print(f"Altitude range: {cluster_c['alt_mean'].min():.1f} — {cluster_c['alt_mean'].max():.1f} km")

Cluster C satellites: 32
Altitude range: 400.2 — 477.4 km


In [55]:
# Filter historical TLEs to cluster C only
cluster_c_norads = cluster_c['norad'].tolist() if 'norad' in cluster_c.columns else [
    str(tles[i]['line1'][2:7].strip()) for i in cluster_c_idx
]

print(f"Cluster C NORADs: {cluster_c_norads}")

# Filter hist_df to cluster C
hist_c = hist_df[hist_df['norad'].isin(cluster_c_norads)].copy()

print(f"Historical records for Cluster C: {len(hist_c)}")
print(f"Unique satellites: {hist_c['norad'].nunique()}")

# Daily TLEs for cluster C
daily_c = (hist_c.sort_values('epoch')
                  .groupby(['norad', 'date'])
                  .first()
                  .reset_index())

unique_days_c = sorted(daily_c['date'].unique())
print(f"Days with data: {len(unique_days_c)}")
print(f"Range: {unique_days_c[0]} → {unique_days_c[-1]}")

Cluster C NORADs: ['58258', '58259', '58262', '58271', '58273', '58274', '58276', '58280', '58282', '58284', '58285', '58286', '58294', '58295', '58299', '58301', '58302', '58304', '58306', '58307', '58308', '58309', '58318', '58320', '58322', '58323', '58326', '58327', '58328', '58336', '58340', '59175']
Historical records for Cluster C: 3514
Unique satellites: 31
Days with data: 44
Range: 2023-11-28 → 2024-01-10


In [57]:
# Pairwise 3D separation for cluster C only
disp_c_results = []

for day in unique_days_c:
    day_tles = daily_c[daily_c['date'] == day]
    day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
    
    if len(day_tles) < 5:
        continue
    
    eval_time = Time(f"{day}T12:00:00", scale='utc')
    jd1, jd2  = eval_time.jd1, eval_time.jd2
    
    positions = []
    for _, row in day_tles.iterrows():
        try:
            sat = Satrec.twoline2rv(row['line1'], row['line2'])
            e, r, v = sat.sgp4(jd1, jd2)
            if e == 0:
                positions.append(np.array(r))
        except:
            continue
    
    if len(positions) < 5:
        continue
    
    positions = np.array(positions)
    
    dists = []
    for a in range(len(positions)):
        for b in range(a+1, len(positions)):
            dists.append(np.linalg.norm(positions[a] - positions[b]))
    
    days_since_launch = (pd.Timestamp(day) - pd.Timestamp('2023-11-11')).days
    
    disp_c_results.append({
        'date':              day,
        'days_since_launch': days_since_launch,
        'n_sats':            len(positions),
        'mean_sep':          np.mean(dists),
        'median_sep':        np.median(dists),
        'min_sep':           np.min(dists),
        'max_sep':           np.max(dists),
    })

disp_c_df = pd.DataFrame(disp_c_results)
initial_mean_c = disp_c_df['mean_sep'].iloc[0]
disp_c_df['rsi'] = disp_c_df['mean_sep'] / initial_mean_c

# Along-track for cluster C
along_c_results = []

for day in unique_days_c:
    day_tles = daily_c[daily_c['date'] == day]
    day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
    
    if len(day_tles) < 5:
        continue
    
    eval_time = Time(f"{day}T12:00:00", scale='utc')
    jd1, jd2  = eval_time.jd1, eval_time.jd2
    
    positions  = []
    velocities = []
    for _, row in day_tles.iterrows():
        try:
            sat = Satrec.twoline2rv(row['line1'], row['line2'])
            e, r, v = sat.sgp4(jd1, jd2)
            if e == 0:
                positions.append(np.array(r))
                velocities.append(np.array(v))
        except:
            continue
    
    if len(positions) < 5:
        continue
    
    positions  = np.array(positions)
    velocities = np.array(velocities)
    
    along_seps = []
    for a in range(len(positions)):
        v_hat = velocities[a] / np.linalg.norm(velocities[a])
        for b in range(a+1, len(positions)):
            dr = positions[b] - positions[a]
            along_seps.append(np.abs(np.dot(dr, v_hat)))
    
    days_since_launch = (pd.Timestamp(day) - pd.Timestamp('2023-11-11')).days
    
    along_c_results.append({
        'date':              day,
        'days_since_launch': days_since_launch,
        'n_sats':            len(positions),
        'mean_along':        np.mean(along_seps),
        'median_along':      np.median(along_seps),
        'min_along':         np.min(along_seps),
    })

along_c_df = pd.DataFrame(along_c_results)
initial_along_c = along_c_df['mean_along'].iloc[0]
along_c_df['rsi'] = along_c_df['mean_along'] / initial_along_c

# RAAN spread for cluster C
raan_c_results = []

for day in unique_days_c:
    day_tles = daily_c[daily_c['date'] == day]
    day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
    
    if len(day_tles) < 5:
        continue
    
    raans = []
    for _, row in day_tles.iterrows():
        try:
            raan = float(row['line2'][17:25].strip())
            raans.append(raan)
        except:
            continue
    
    raans = np.array(raans)
    raan_diffs = []
    for a in range(len(raans)):
        for b in range(a+1, len(raans)):
            diff = abs(raans[a] - raans[b])
            if diff > 180:
                diff = 360 - diff
            raan_diffs.append(diff)
    
    days_since_launch = (pd.Timestamp(day) - pd.Timestamp('2023-11-11')).days
    
    raan_c_results.append({
        'date':              day,
        'days_since_launch': days_since_launch,
        'n_sats':            len(raans),
        'mean_raan_diff':    np.mean(raan_diffs),
        'median_raan_diff':  np.median(raan_diffs),
        'min_raan_diff':     np.min(raan_diffs),
    })

raan_c_df = pd.DataFrame(raan_c_results)
initial_raan_c = raan_c_df['mean_raan_diff'].iloc[0]
raan_c_df['rsi'] = raan_c_df['mean_raan_diff'] / initial_raan_c

# Combined plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=disp_c_df['days_since_launch'],
    y=disp_c_df['rsi'],
    mode='lines+markers',
    name='3D Separation RSI',
    line=dict(color='cyan', width=2)
))

fig.add_trace(go.Scatter(
    x=along_c_df['days_since_launch'],
    y=along_c_df['rsi'],
    mode='lines+markers',
    name='Along-Track RSI',
    line=dict(color='orange', width=2)
))

fig.add_trace(go.Scatter(
    x=raan_c_df['days_since_launch'],
    y=raan_c_df['rsi'],
    mode='lines+markers',
    name='RAAN Spread RSI',
    line=dict(color='magenta', width=2)
))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='Baseline', annotation_font_color='gray')

fig.update_layout(
    title='Cluster C — Separation RSI Comparison Over 60 Days',
    xaxis_title='Days Since Launch',
    yaxis_title='RSI (normalized to day 17)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)

fig.show()

print(f"\nDay 17 → Day 60 RSI growth:")
print(f"   3D Separation:  {disp_c_df['rsi'].iloc[0]:.3f} → {disp_c_df['rsi'].iloc[-1]:.3f}")
print(f"   Along-Track:    {along_c_df['rsi'].iloc[0]:.3f} → {along_c_df['rsi'].iloc[-1]:.3f}")
print(f"   RAAN Spread:    {raan_c_df['rsi'].iloc[0]:.3f} → {raan_c_df['rsi'].iloc[-1]:.3f}")


Day 17 → Day 60 RSI growth:
   3D Separation:  1.000 → 2.143
   Along-Track:    1.000 → 1.128
   RAAN Spread:    1.000 → 2.422


In [58]:
# RTN decomposition for Cluster C
rtn_results = []

for day in unique_days_c:
    day_tles = daily_c[daily_c['date'] == day]
    day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
    
    if len(day_tles) < 5:
        continue
    
    eval_time = Time(f"{day}T12:00:00", scale='utc')
    jd1, jd2  = eval_time.jd1, eval_time.jd2
    
    positions  = []
    velocities = []
    
    for _, row in day_tles.iterrows():
        try:
            sat = Satrec.twoline2rv(row['line1'], row['line2'])
            e, r, v = sat.sgp4(jd1, jd2)
            if e == 0:
                positions.append(np.array(r))
                velocities.append(np.array(v))
        except:
            continue
    
    if len(positions) < 5:
        continue
    
    positions  = np.array(positions)
    velocities = np.array(velocities)
    
    r_seps = []
    t_seps = []
    n_seps = []
    
    for a in range(len(positions)):
        # RTN frame for reference satellite a
        R_hat = positions[a]  / np.linalg.norm(positions[a])
        N_hat = np.cross(positions[a], velocities[a])
        N_hat = N_hat / np.linalg.norm(N_hat)
        T_hat = np.cross(N_hat, R_hat)
        T_hat = T_hat / np.linalg.norm(T_hat)
        
        for b in range(a+1, len(positions)):
            dr = positions[b] - positions[a]
            r_seps.append(np.abs(np.dot(dr, R_hat)))
            t_seps.append(np.abs(np.dot(dr, T_hat)))
            n_seps.append(np.abs(np.dot(dr, N_hat)))
    
    days_since_launch = (pd.Timestamp(day) - pd.Timestamp('2023-11-11')).days
    
    rtn_results.append({
        'date':              day,
        'days_since_launch': days_since_launch,
        'n_sats':            len(positions),
        'mean_R':            np.mean(r_seps),
        'mean_T':            np.mean(t_seps),
        'mean_N':            np.mean(n_seps),
        'median_R':          np.median(r_seps),
        'median_T':          np.median(t_seps),
        'median_N':          np.median(n_seps),
    })

rtn_df = pd.DataFrame(rtn_results)

# Normalize to day 17
rtn_df['rsi_R'] = rtn_df['mean_R'] / rtn_df['mean_R'].iloc[0]
rtn_df['rsi_T'] = rtn_df['mean_T'] / rtn_df['mean_T'].iloc[0]
rtn_df['rsi_N'] = rtn_df['mean_N'] / rtn_df['mean_N'].iloc[0]

# Plot RSI
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'], y=rtn_df['rsi_R'],
    mode='lines+markers', name='Radial RSI',
    line=dict(color='red', width=2)
))
fig.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'], y=rtn_df['rsi_T'],
    mode='lines+markers', name='Along-Track RSI',
    line=dict(color='orange', width=2)
))
fig.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'], y=rtn_df['rsi_N'],
    mode='lines+markers', name='Cross-Track RSI',
    line=dict(color='magenta', width=2)
))
fig.add_hline(y=1.0, line_dash='dash', line_color='gray')

fig.update_layout(
    title='Cluster C — RTN Separation RSI Over 60 Days',
    xaxis_title='Days Since Launch',
    yaxis_title='RSI (normalized to day 17)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)
fig.show()

# Plot absolute km
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'], y=rtn_df['mean_R'],
    mode='lines+markers', name='Radial (km)',
    line=dict(color='red', width=2)
))
fig2.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'], y=rtn_df['mean_T'],
    mode='lines+markers', name='Along-Track (km)',
    line=dict(color='orange', width=2)
))
fig2.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'], y=rtn_df['mean_N'],
    mode='lines+markers', name='Cross-Track (km)',
    line=dict(color='magenta', width=2)
))

fig2.update_layout(
    title='Cluster C — RTN Absolute Separation Over 60 Days',
    xaxis_title='Days Since Launch',
    yaxis_title='Mean Separation (km)',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(gridcolor='gray'),
    yaxis=dict(gridcolor='gray'),
    legend=dict(bgcolor='black')
)
fig2.show()

print(f"\nRTN Separation — Day 17 → Day 60:")
print(f"{'Component':<15} {'Day 17 (km)':>12} {'Day 60 (km)':>12} {'RSI Growth':>12}")
print("-" * 55)
print(f"{'Radial':<15} {rtn_df['mean_R'].iloc[0]:>12.2f} {rtn_df['mean_R'].iloc[-1]:>12.2f} {rtn_df['rsi_R'].iloc[-1]:>12.3f}")
print(f"{'Along-Track':<15} {rtn_df['mean_T'].iloc[0]:>12.2f} {rtn_df['mean_T'].iloc[-1]:>12.2f} {rtn_df['rsi_T'].iloc[-1]:>12.3f}")
print(f"{'Cross-Track':<15} {rtn_df['mean_N'].iloc[0]:>12.2f} {rtn_df['mean_N'].iloc[-1]:>12.2f} {rtn_df['rsi_N'].iloc[-1]:>12.3f}")


RTN Separation — Day 17 → Day 60:
Component        Day 17 (km)  Day 60 (km)   RSI Growth
-------------------------------------------------------
Radial               1796.04      7113.87        3.961
Along-Track          3594.16      4054.06        1.128
Cross-Track             1.69         6.91        4.096


In [59]:
# Radial growth rate per day — derivative of radial separation curve
# Use finite difference on the mean_R values

rtn_df['radial_growth_rate'] = np.gradient(rtn_df['mean_R'], rtn_df['days_since_launch'])

# Plot radial separation and growth rate together
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=('Mean Radial Separation (km)',
                                    'Radial Growth Rate (km/day)'))

# Top — absolute radial separation
fig.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'],
    y=rtn_df['mean_R'],
    mode='lines+markers',
    name='Mean Radial Sep',
    line=dict(color='red', width=2)
), row=1, col=1)

# Bottom — growth rate
fig.add_trace(go.Scatter(
    x=rtn_df['days_since_launch'],
    y=rtn_df['radial_growth_rate'],
    mode='lines+markers',
    name='Growth Rate (km/day)',
    line=dict(color='orange', width=2)
), row=2, col=1)

# Mark peak growth rate
peak_idx  = rtn_df['radial_growth_rate'].idxmax()
peak_day  = rtn_df.loc[peak_idx, 'days_since_launch']
peak_rate = rtn_df.loc[peak_idx, 'radial_growth_rate']

fig.add_vline(x=peak_day, line_dash='dash', line_color='cyan',
              annotation_text=f'Peak growth day {peak_day}',
              annotation_font_color='cyan')

# Mark inflection point — where growth rate drops below 50% of peak
half_peak = peak_rate * 0.5
below_half = rtn_df[rtn_df['radial_growth_rate'] < half_peak]
if len(below_half) > 0:
    inflection_day = below_half.iloc[0]['days_since_launch']
    fig.add_vline(x=inflection_day, line_dash='dash', line_color='yellow',
                  annotation_text=f'50% decay day {inflection_day}',
                  annotation_font_color='yellow')

fig.update_layout(
    title='Cluster C — Radial Separation Growth Rate Analysis',
    paper_bgcolor='black',
    plot_bgcolor='black',
    font=dict(color='white'),
    legend=dict(bgcolor='black')
)
fig.update_xaxes(gridcolor='gray', title_text='Days Since Launch')
fig.update_yaxes(gridcolor='gray')

fig.show()

print(f"Radial Growth Rate Analysis:")
print(f"{'Days':>6} {'Mean R (km)':>12} {'Growth Rate (km/day)':>22}")
print("-" * 45)
for _, row in rtn_df.iterrows():
    print(f"{row['days_since_launch']:>6} {row['mean_R']:>12.1f} {row['radial_growth_rate']:>22.2f}")

print(f"\nKey findings:")
print(f"   Peak growth rate:     {peak_rate:.2f} km/day at day {peak_day}")
print(f"   50% decay point:      day {inflection_day}")
print(f"   Optimal ID window:    days {rtn_df['days_since_launch'].iloc[0]} — {inflection_day}")

Radial Growth Rate Analysis:
  Days  Mean R (km)   Growth Rate (km/day)
---------------------------------------------
    17       1796.0                 218.52
    18       2014.6                -581.19
    19        633.7                 183.02
    20       2380.6                 974.73
    21       2583.1                 207.40
    22       2795.4                 214.78
    23       3012.7                 216.93
    24       3229.2                 213.74
    25       3440.2                 206.42
    26       3642.1                 198.78
    27       3837.7                 214.66
    28       4071.4                 218.21
    29       4274.1                 187.70
    30       4446.8                 186.64
    31       4647.4                 194.43
    32       4835.6                 180.84
    33       5009.1                 168.32
    34       5172.3                 162.34
    35       5333.8                 163.03
    36       5498.3                 155.27
    37       5644.3   

In [62]:
# Only use days with sufficient satellite coverage
min_sats = 28  # threshold for full catalog
rtn_clean = rtn_df[rtn_df['n_sats'] >= min_sats].copy()
rtn_clean['radial_growth_rate'] = np.gradient(rtn_clean['mean_R'], 
                                               rtn_clean['days_since_launch'])

peak_idx  = rtn_clean['radial_growth_rate'].idxmax()
peak_day  = rtn_clean.loc[peak_idx, 'days_since_launch']
peak_rate = rtn_clean.loc[peak_idx, 'radial_growth_rate']

half_peak    = peak_rate * 0.5
below_half   = rtn_clean[rtn_clean['radial_growth_rate'] < half_peak]
inflection_day = below_half.iloc[0]['days_since_launch'] if len(below_half) > 0 else None

print(f"Clean dataset — {len(rtn_clean)} days (≥{min_sats} satellites)")
print(f"Peak growth rate: {peak_rate:.2f} km/day at day {peak_day}")
print(f"50% decay point:  day {inflection_day}")
print(f"Optimal ID window: days {rtn_clean['days_since_launch'].iloc[0]} — {inflection_day}")

Clean dataset — 43 days (≥28 satellites)
Peak growth rate: 218.52 km/day at day 17
50% decay point:  day 41
Optimal ID window: days 17 — 41
